In [10]:
# load packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

import geopandas as gpd

# load clean data — baseline feature set
CRMLSSold_cleaned = pd.read_csv('../data/cleaned_CRMLSSOLD_baseline.csv')
print(CRMLSSold_cleaned.info())
CRMLSSold_cleaned.head()

<class 'pandas.DataFrame'>
RangeIndex: 79306 entries, 0 to 79305
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CloseDate              79306 non-null  str    
 1   ClosePrice             79306 non-null  float64
 2   LivingArea             79306 non-null  float64
 3   BedroomsTotal          79306 non-null  int64  
 4   BathroomsTotalInteger  79306 non-null  int64  
 5   YearBuilt              79306 non-null  int64  
 6   GarageSpaces           76469 non-null  float64
 7   LotSizeSquareFeet      77911 non-null  float64
 8   CountyOrParish         79306 non-null  str    
 9   Latitude               79306 non-null  float64
 10  Longitude              79306 non-null  float64
 11  PoolPrivateYN          79306 non-null  str    
 12  ViewYN                 79306 non-null  str    
 13  FireplaceYN            79306 non-null  str    
 14  NewConstructionYN      79306 non-null  str    
 15  HasAssociatio

,CloseDate,ClosePrice,LivingArea,BedroomsTotal,BathroomsTotalInteger,YearBuilt,GarageSpaces,LotSizeSquareFeet,CountyOrParish,Latitude,Longitude,PoolPrivateYN,ViewYN,FireplaceYN,NewConstructionYN,HasAssociationFee
0,2025-11-30,1250000.0,1027.0,3,2,1961,2.0,5913.0,Orange,33.676050,-117.995210,False,False,False,False,False
1,2025-11-20,2299995.0,1129.0,3,1,1949,2.0,18432.0,Santa Clara,37.260693,-121.934121,Unknown,False,True,False,Unknown
2,2025-11-26,810000.0,1619.0,4,3,1978,2.0,5300.0,San Diego,32.564997,-117.064393,False,False,True,Unknown,False
3,2025-11-17,925000.0,2872.0,5,3,2000,3.0,5272.0,San Diego,32.575778,-117.024433,False,False,True,Unknown,True
4,2025-11-25,1300000.0,1727.0,3,2,1950,3.0,10500.0,San Luis Obispo,35.553064,-120.708510,False,False,True,False,False


### train/test split function (same time-based split as 03/04/05)

In [11]:
# Same time-based train/test split as 03/04/05 — test = most recent month, train = all prior months
def make_train_test_split(df, feature_cols, training_months=None):
    df['CloseDate'] = pd.to_datetime(df['CloseDate'])
    df['CloseMonth'] = df['CloseDate'].dt.to_period('M')

    test_month  = df['CloseMonth'].max()
    train_start = test_month - training_months if training_months else df['CloseMonth'].min()

    train = df[(df['CloseMonth'] >= train_start) & (df['CloseMonth'] < test_month)].copy()
    test  = df[df['CloseMonth'] == test_month].copy()

    X_train = train[feature_cols]
    y_train = train['ClosePrice']
    X_test  = test[feature_cols]
    y_test  = test['ClosePrice']

    print(f"Test month:  {test_month}")
    print(f"Train range: {train['CloseMonth'].min()} to {train['CloseMonth'].max()}  ({train['CloseMonth'].nunique()} months)")
    print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

    return X_train, y_train, X_test, y_test

### Location features (PostalCode, City, DistrictName, MLSAreaMajor) — same join as 04/05

All six models will share one feature set (the richest one, from `05_advanced_models.ipynb`), so we pull in the location fields once here.

In [12]:
# Base ("old") feature/column groups — same as 03/04/05
base_feature_cols = [col for col in CRMLSSold_cleaned.columns if col != 'ClosePrice']
categorical_cols = ['PoolPrivateYN', 'ViewYN', 'FireplaceYN', 'NewConstructionYN',
                    'HasAssociationFee', 'CountyOrParish']
numeric_cols = ['LivingArea', 'LotSizeSquareFeet', 'BedroomsTotal', 'BathroomsTotalInteger',
                'YearBuilt', 'GarageSpaces', 'Latitude', 'Longitude']

# Pull PostalCode/City/MLSAreaMajor from the all-features CSV (row-aligned with CRMLSSold_cleaned)
_all_features_df = pd.read_csv('../data/cleaned_CRMLSSOLD_all_features.csv')
assert len(_all_features_df) == len(CRMLSSold_cleaned), "Row count mismatch between CSVs"
CRMLSSold_cleaned['PostalCode'] = _all_features_df['PostalCode'].astype(str)
CRMLSSold_cleaned['City'] = _all_features_df['City']
CRMLSSold_cleaned['MLSAreaMajor'] = _all_features_df['MLSAreaMajor']

# School district join — same approach as 04/05
districts = gpd.read_file('../data/ca_school_districts.geojson')[['DistrictName', 'geometry']]
listings = gpd.GeoDataFrame(
    CRMLSSold_cleaned,
    geometry=gpd.points_from_xy(CRMLSSold_cleaned['Longitude'], CRMLSSold_cleaned['Latitude']),
    crs='EPSG:4326'
)

n_before_join = len(listings)
listings = gpd.sjoin(listings, districts, how='left', predicate='within')
listings = listings[~listings.index.duplicated(keep='first')]
listings = listings.drop(columns=['geometry', 'index_right'])
assert len(listings) == n_before_join, "Row count changed after dedup — join still producing extras"

n_unmatched = listings['DistrictName'].isna().sum()
print(f"Unmatched to a district: {n_unmatched:,} ({n_unmatched / len(listings):.2%})")
listings = listings.dropna(subset=['DistrictName'])

CRMLSSold_cleaned = pd.DataFrame(listings)

# Keep ClosePrice in dollars for MAPE/MdAPE later; log-transform into a
# separate column for model fitting/R^2 (comparable to 03/04/05, which log-transformed in place)
CRMLSSold_cleaned['ClosePriceDollars'] = CRMLSSold_cleaned['ClosePrice']
CRMLSSold_cleaned['ClosePrice'] = np.log(CRMLSSold_cleaned['ClosePrice'])

print(f"Unique PostalCode: {CRMLSSold_cleaned['PostalCode'].nunique():,}")
print(f"Unique City: {CRMLSSold_cleaned['City'].nunique():,}")
print(f"Unique DistrictName: {CRMLSSold_cleaned['DistrictName'].nunique():,}")
print(f"Unique MLSAreaMajor: {CRMLSSold_cleaned['MLSAreaMajor'].nunique():,}")

Unmatched to a district: 5 (0.01%)
Unique PostalCode: 1,295
Unique City: 880
Unique DistrictName: 545
Unique MLSAreaMajor: 963


### Feature set & preprocessor — same feature set for every model

All six models use the same feature set for a like-for-like comparison: base features + `PostalCode`/`City`/`DistrictName`/`MLSAreaMajor` — the richest feature set from `05_advanced_models.ipynb` (used there for XGBoost/LightGBM/CatBoost).

In [13]:
# Shared feature set: base features + PostalCode/City/DistrictName/MLSAreaMajor
location_cols = ['PostalCode', 'City', 'DistrictName', 'MLSAreaMajor']
feature_cols = base_feature_cols + location_cols

preprocessor = ColumnTransformer([
    ('impute', SimpleImputer(strategy='median'), numeric_cols),
    ('encode', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('encode_location', OneHotEncoder(handle_unknown='ignore', min_frequency=50), location_cols),
])

X_train, y_train, X_test, y_test = make_train_test_split(
    CRMLSSold_cleaned, feature_cols, training_months=None
)
# dollar-scale targets for MAPE/MdAPE (ClosePrice above is log-transformed)
y_train_dollars = CRMLSSold_cleaned.loc[X_train.index, 'ClosePriceDollars']
y_test_dollars = CRMLSSold_cleaned.loc[X_test.index, 'ClosePriceDollars']

Test month:  2026-06
Train range: 2025-11 to 2026-05  (7 months)
X_train: (67174, 19)  |  X_test: (12127, 19)


### Models with their best parameters from 04/05

- `LinearRegression` — no hyperparameters
- `DecisionTree` / `RandomForest` — fixed params from `04_model_comparison.ipynb`
- `XGBoost` / `LightGBM` / `CatBoost` — grid-search winners from `05_advanced_models.ipynb` (`n_estimators`/`iterations`=1000, `learning_rate`=0.1, `max_depth`/`depth`=8)

In [14]:
models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42, max_depth=12, min_samples_leaf=10),
    'RandomForest': RandomForestRegressor(random_state=42, max_depth=16, min_samples_leaf=5, n_estimators=200),
    'XGBoost': XGBRegressor(random_state=42, n_estimators=1000, learning_rate=0.1, max_depth=8),
    'LightGBM': LGBMRegressor(random_state=42, verbose=-1, n_estimators=1000, learning_rate=0.1, max_depth=8),
    'CatBoost': CatBoostRegressor(random_state=42, verbose=False, iterations=1000, learning_rate=0.1, depth=8),
}

### Fit each model and compute R^2, MAPE, MdAPE

Every model was trained on `log(ClosePrice)`, so R^2 is reported in log-price space (comparable to 03/04/05). Predictions are converted back to dollars (`np.exp`) before computing MAPE/MdAPE against the true `ClosePrice` in dollars.

Metrics are computed on both train and test sets so we can check the train/test gap for overfitting — a large gap (high train R^2, much lower test R^2) means the model is memorizing training data rather than generalizing.

In [15]:
def median_absolute_percentage_error(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true))

eval_results = {}
for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)

    preds_log_train = pipeline.predict(X_train)
    preds_log_test = pipeline.predict(X_test)

    r2_train = r2_score(y_train, preds_log_train)
    r2_test = r2_score(y_test, preds_log_test)

    preds_dollars_train = np.exp(preds_log_train)
    preds_dollars_test = np.exp(preds_log_test)

    mape_train = mean_absolute_percentage_error(y_train_dollars, preds_dollars_train)
    mape_test = mean_absolute_percentage_error(y_test_dollars, preds_dollars_test)

    mdape_train = median_absolute_percentage_error(y_train_dollars, preds_dollars_train)
    mdape_test = median_absolute_percentage_error(y_test_dollars, preds_dollars_test)

    eval_results[name] = {
        'R^2': r2_test, 'MAPE': mape_test, 'MdAPE': mdape_test,
        'R^2 (train)': r2_train, 'MAPE (train)': mape_train, 'MdAPE (train)': mdape_train,
        'R^2 gap': r2_train - r2_test, 'MAPE gap': mape_test - mape_train, 'MdAPE gap': mdape_test - mdape_train,
    }
    print(f"{name}")
    print(f"  R^2:    test {r2_test:.4f}  |  train {r2_train:.4f}  |  gap {r2_train - r2_test:.4f}")
    print(f"  MAPE:   test {mape_test:.4f}  |  train {mape_train:.4f}  |  gap {mape_test - mape_train:.4f}")
    print(f"  MdAPE:  test {mdape_test:.4f}  |  train {mdape_train:.4f}  |  gap {mdape_test - mdape_train:.4f}\n")

LinearRegression
  R^2:    test 0.8528  |  train 0.8502  |  gap -0.0026
  MAPE:   test 0.1704  |  train 0.1723  |  gap -0.0019
  MdAPE:  test 0.1276  |  train 0.1269  |  gap 0.0007

DecisionTree
  R^2:    test 0.8432  |  train 0.8639  |  gap 0.0207
  MAPE:   test 0.1718  |  train 0.1601  |  gap 0.0117
  MdAPE:  test 0.1154  |  train 0.1097  |  gap 0.0058

RandomForest
  R^2:    test 0.9112  |  train 0.9432  |  gap 0.0320
  MAPE:   test 0.1269  |  train 0.1010  |  gap 0.0258
  MdAPE:  test 0.0863  |  train 0.0691  |  gap 0.0172

XGBoost
  R^2:    test 0.9323  |  train 0.9696  |  gap 0.0373
  MAPE:   test 0.1123  |  train 0.0758  |  gap 0.0365
  MdAPE:  test 0.0796  |  train 0.0541  |  gap 0.0255

LightGBM
  R^2:    test 0.9333  |  train 0.9499  |  gap 0.0166
  MAPE:   test 0.1124  |  train 0.0978  |  gap 0.0146
  MdAPE:  test 0.0799  |  train 0.0699  |  gap 0.0100

CatBoost
  R^2:    test 0.9335  |  train 0.9455  |  gap 0.0120
  MAPE:   test 0.1124  |  train 0.1036  |  gap 0.0088
  MdAP

### LightGBM & CatBoost with native categorical handling

The comparison above one-hot encodes every model equally for a fair baseline. But LightGBM and CatBoost both have native categorical support that's usually a better fit than one-hot for high-cardinality columns like `PostalCode` (1,295 uniques), `City` (880), `DistrictName` (545), and `MLSAreaMajor` (963). One split can do what one-hot needs many sparse columns to approximate.

Here we refit  those two models on the raw (unencoded) categorical columns, letting each library handle categories its own way, to see if it beats their one-hot results above. This represents "best possible" tuning for these two rather than an apples-to-apples comparison.

In [ ]:
# Raw (unencoded) categorical columns — restrict to the columns the one-hot preprocessor
# actually used (numeric_cols + all_cat_cols). feature_cols also carries CloseDate, which the
# ColumnTransformer silently dropped for the other six models but would break a raw .fit() here.
all_cat_cols = categorical_cols + location_cols
native_feature_cols = numeric_cols + all_cat_cols

X_train_native = X_train[native_feature_cols].copy()
X_test_native = X_test[native_feature_cols].copy()
for col in all_cat_cols:
    # CatBoost's cat_features can't hold NaN, so route true missing values AND categories
    # unseen in train (e.g. a test-only PostalCode) into the same explicit 'Unknown' bucket
    # used throughout 02_preprocessing.ipynb for other categorical columns.
    train_values = X_train_native[col].astype(str).where(X_train_native[col].notna(), 'Unknown')
    train_categories = pd.Index(train_values.unique())
    if 'Unknown' not in train_categories:
        train_categories = train_categories.insert(0, 'Unknown')

    test_values = X_test_native[col].astype(str).where(X_test_native[col].notna(), 'Unknown')
    test_values = test_values.where(test_values.isin(train_categories), 'Unknown')

    X_train_native[col] = pd.Categorical(train_values, categories=train_categories)
    X_test_native[col] = pd.Categorical(test_values, categories=train_categories)

native_models = {
    'LightGBM (native cat)': LGBMRegressor(random_state=42, verbose=-1, n_estimators=1000, learning_rate=0.1, max_depth=8),
    'CatBoost (native cat)': CatBoostRegressor(random_state=42, verbose=False, iterations=1000, learning_rate=0.1, depth=8,
                                                cat_features=all_cat_cols),
}

for name, model in native_models.items():
    model.fit(X_train_native, y_train)

    preds_log_train = model.predict(X_train_native)
    preds_log_test = model.predict(X_test_native)

    r2_train = r2_score(y_train, preds_log_train)
    r2_test = r2_score(y_test, preds_log_test)

    preds_dollars_train = np.exp(preds_log_train)
    preds_dollars_test = np.exp(preds_log_test)

    mape_train = mean_absolute_percentage_error(y_train_dollars, preds_dollars_train)
    mape_test = mean_absolute_percentage_error(y_test_dollars, preds_dollars_test)

    mdape_train = median_absolute_percentage_error(y_train_dollars, preds_dollars_train)
    mdape_test = median_absolute_percentage_error(y_test_dollars, preds_dollars_test)

    eval_results[name] = {
        'R^2': r2_test, 'MAPE': mape_test, 'MdAPE': mdape_test,
        'R^2 (train)': r2_train, 'MAPE (train)': mape_train, 'MdAPE (train)': mdape_train,
        'R^2 gap': r2_train - r2_test, 'MAPE gap': mape_test - mape_train, 'MdAPE gap': mdape_test - mdape_train,
    }
    print(f"{name}")
    print(f"  R^2:    test {r2_test:.4f}  |  train {r2_train:.4f}  |  gap {r2_train - r2_test:.4f}")
    print(f"  MAPE:   test {mape_test:.4f}  |  train {mape_train:.4f}  |  gap {mape_test - mape_train:.4f}")
    print(f"  MdAPE:  test {mdape_test:.4f}  |  train {mdape_train:.4f}  |  gap {mdape_test - mdape_train:.4f}\n")

### Full comparison table (one-hot baseline + native-categorical LightGBM/CatBoost)

In [19]:
column_order = ['R^2', 'R^2 (train)', 'R^2 gap', 'MAPE', 'MAPE (train)', 'MAPE gap', 'MdAPE', 'MdAPE (train)', 'MdAPE gap']
full_comparison_table = pd.DataFrame(eval_results).T.sort_values('R^2', ascending=False)
full_comparison_table[column_order].round(4)

,R^2,R^2 (train),R^2 gap,MAPE,MAPE (train),MAPE gap,MdAPE,MdAPE (train),MdAPE gap
LightGBM (native cat),0.9360,0.9695,0.0335,0.1092,0.0770,0.0323,0.0772,0.0564,0.0208
CatBoost (native cat),0.9354,0.9500,0.0145,0.1099,0.0985,0.0114,0.0778,0.0709,0.0069
CatBoost,0.9335,0.9455,0.0120,0.1124,0.1036,0.0088,0.0804,0.0753,0.0052
LightGBM,0.9333,0.9499,0.0166,0.1124,0.0978,0.0146,0.0799,0.0699,0.0100
XGBoost,0.9323,0.9696,0.0373,0.1123,0.0758,0.0365,0.0796,0.0541,0.0255
RandomForest,0.9112,0.9432,0.0320,0.1269,0.1010,0.0258,0.0863,0.0691,0.0172
LinearRegression,0.8528,0.8502,-0.0026,0.1704,0.1723,-0.0019,0.1276,0.1269,0.0007
DecisionTree,0.8432,0.8639,0.0207,0.1718,0.1601,0.0117,0.1154,0.1097,0.0058


In [20]:
full_comparison_table.to_csv('../metric_summary/metrics_summary.csv')